In [1]:
# Parameters
BATCH_MODE = True


# Docteur vs ChatGPT : Chatbot medical multi-agent

**Durée estimée** : 30 minutes | **Prérequis** : compte OpenAI avec clé API, connaissances de base en Python

Dans ce cas d'usage, nous construisons un systeme de consultation medicale simulee ou trois agents IA cooperent : un **medecin generaliste** qui pose des questions, une **IA medicale** qui analyse les symptomes, et un **pharmacien** qui recommande des traitements. Le dialogue est orchestre par Semantic Kernel via `AgentGroupChat`, avec une strategie de terminaison automatique.

> **Avertissement** : ce notebook est un exercice pedagogique. Les diagnostics et recommandations medicamenteuses sont simules et ne remplacent en aucun cas une consultation medicale reelle.

In [2]:
%pip install semantic-kernel openai python-dotenv --quiet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import des bibliothèques


Ce cas d'usage met en oeuvre une **conversation multi-agent** dans le domaine medical. Trois agents cooperent : un medecin generaliste, une IA specialisee en diagnostic et un pharmacien. Chaque agent dispose de son propre prompt systeme et de plugins dedies via des `@kernel_function`.

**Objectifs pedagogiques** :
- Comprendre l'orchestration multi-agent avec `AgentGroupChat`
- Utiliser des plugins `@kernel_function` pour doter les agents de capacites specifiques
- Implementer une `TerminationStrategy` pour controler la fin du dialogue
- Configurer `FunctionChoiceBehavior.Auto()` pour l'appel automatique de plugins

In [3]:
import os
import logging
import asyncio
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.agents.strategies import TerminationStrategy
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function, KernelArguments
from semantic_kernel.connectors.ai import FunctionChoiceBehavior
from typing import Annotated

In [4]:
# Charger les variables d'environnement
load_dotenv()

True

## Configuration des logs

Le module `logging` permet de tracer les echanges entre agents en temps reel. Chaque message sera horodate et identifie par le nom de l'agent emetteur, ce qui facilite le debug des conversations multi-agent.

In [5]:
# Configuration des logs
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger('MedicalAI')

## Création du kernel Semantic Kernel

In [6]:
# Création du kernel Semantic Kernel
def create_kernel():
    kernel = Kernel()
    kernel.add_service(OpenAIChatCompletion(
        service_id="openai",
        ai_model_id="gpt-4o-mini",  # Modifier si besoin
        api_key=os.getenv("OPENAI_API_KEY")
    ))
    return kernel

## Plugins medicaux : doter les agents de competences specifiques

Chaque agent dispose d'un plugin dedie contenant des fonctions annotees avec `@kernel_function`. Ces fonctions agissent comme des outils que le LLM peut invoquer automatiquement lorsqu'il en a besoin :

- **DoctorPlugin** : pose des questions de suivi en fonction du symptome mentionne
- **MedicalAIPlugin** : evalue la gravite d'un symptome (Leger, Modere, Severe, Critique)
- **PharmacistPlugin** : recommande un medicament adapte avec posologie et precautions

Le kernel est l'element central de Semantic Kernel. La fonction `create_kernel()` instancie un kernel et y enregistre un service `OpenAIChatCompletion` qui fournira l'acces au modele GPT-4o-mini. Chaque agent utilisera ce kernel partage pour generer ses reponses.

In [7]:
class DoctorPlugin:
    """Plugin permettant au médecin de poser des questions complémentaires sur les symptômes."""
    @kernel_function(description="Pose des questions supplémentaires pour affiner le diagnostic.")
    def ask_followup_questions(self, symptom: Annotated[str, "Symptôme décrit par l'utilisateur"]) -> str:
        """Retourne une question en fonction du symptôme mentionné."""
        questions_map = {
            "fièvre": "Depuis combien de temps avez-vous de la fièvre ?",
            "maux de tête": "Avez-vous une sensibilité à la lumière ou au bruit ?",
            "douleur thoracique": "La douleur est-elle aiguë ou diffuse ?",
        }
        return questions_map.get(symptom.lower(), "Pouvez-vous donner plus de détails sur vos symptômes ?")

class MedicalAIPlugin:
    """Plugin qui analyse la gravité des symptômes."""
    @kernel_function(description="Vérifie la gravité d'un symptôme médical.")
    def check_symptom_severity(self, symptom: Annotated[str, "Symptôme décrit par l'utilisateur"]) -> str:
        """Retourne une évaluation de la gravité du symptôme."""
        severity_map = {
            "fièvre": "Modérée",
            "maux de tête": "Léger",
            "douleur thoracique": "Sévère",
            "perte de connaissance": "Critique"
        }
        return severity_map.get(symptom.lower(), "Inconnu - consultez un médecin.")

class PharmacistPlugin:
    """Plugin qui recommande des médicaments en fonction du diagnostic."""
    @kernel_function(description="Recommande un médicament adapté à un symptôme.")
    def recommend_medication(self, symptom: Annotated[str, "Symptôme décrit par l'utilisateur"]) -> str:
        """Retourne une suggestion de médicament (avec précautions)."""
        medication_map = {
            "fièvre": "Paracétamol (500mg, toutes les 6h, max 3 jours)",
            "maux de tête": "Ibuprofène (200mg, toutes les 8h, avec précaution si problème gastrique)",
            "douleur thoracique": "Aucun médicament recommandé - Consultez un médecin immédiatement",
        }
        return medication_map.get(symptom.lower(), "Aucun médicament recommandé - Consultez un pharmacien.")


## Création du Kernel

In [8]:
# Création du kernel
kernel = create_kernel()


# Ajout des plugins pour chaque agent

In [9]:
# Ajout des plugins pour chaque agent
kernel.add_plugin(DoctorPlugin(), plugin_name="doctor")
kernel.add_plugin(MedicalAIPlugin(), plugin_name="medical")
kernel.add_plugin(PharmacistPlugin(), plugin_name="pharmacist")


KernelPlugin(name='pharmacist', description=None, functions={'recommend_medication': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='recommend_medication', plugin_name='pharmacist', description='Recommande un médicament adapté à un symptôme.', parameters=[KernelParameterMetadata(name='symptom', description="Symptôme décrit par l'utilisateur", default_value=None, type_='str', is_required=True, type_object=<class 'str'>, schema_data={'type': 'string', 'description': "Symptôme décrit par l'utilisateur"}, include_in_function_choices=True)], is_prompt=False, is_asynchronous=False, return_parameter=KernelParameterMetadata(name='return', description='', default_value=None, type_='str', is_required=True, type_object=<class 'str'>, schema_data={'type': 'string'}, include_in_function_choices=True), additional_properties={}), invocation_duration_histogram=<opentelemetry.metrics._internal.instrument._ProxyHistogram object at 0x00000266AE144050>, streaming_duration_histogram=<opentel

## Prompts systeme : definir le comportement de chaque agent

Les prompts systeme sont les instructions qui guident le comportement de chaque agent. Ils etablissent le role, les contraintes et le style de reponse attendu. Une bonne conception de prompt est essentielle pour eviter les comportements indesirables (diagnostic premature, recommandation non fondee).

In [10]:
DOCTOR_PROMPT = """
Vous êtes un médecin généraliste. Vous posez d'abord des questions pour mieux comprendre les symptômes de l'utilisateur,
puis vous donnez un diagnostic probable basé sur votre expertise médicale. 
Ne donnez jamais de diagnostic sans avoir recueilli assez d'informations.
"""

AI_MEDICAL_PROMPT = """
Vous êtes une IA médicale spécialisée en diagnostic. Analysez les symptômes fournis et proposez un diagnostic basé sur des statistiques et des études médicales. 
Soyez clair et donnez plusieurs hypothèses si nécessaire.
"""

PHARMACIST_PROMPT = """
Vous êtes un pharmacien qualifié. En fonction du diagnostic fourni, vous recommandez les médicaments appropriés. 
Mentionnez toujours les précautions d'utilisation et la nécessité d'une consultation médicale avant la prise de médicaments.
"""


## Création des agents

Les trois prompts systeme definissent le comportement et les contraintes de chaque agent. Le medecin doit poser des questions avant de diagnostiquer, l'IA medicale analyse les symptomes avec une approche statistique, et le pharmacien recommande des traitements en rappelant les precautions d'usage. Cette separation des roles est fondamentale dans une architecture multi-agent.

In [11]:
# Création des agents
doctor_agent = ChatCompletionAgent(
    kernel=kernel,
    name="Docteur_Humain",
    instructions=DOCTOR_PROMPT,
)

ai_medical_agent = ChatCompletionAgent(
    kernel=kernel,
    name="IA_Medicale",
    instructions=AI_MEDICAL_PROMPT,
)

## Configuration pour que l'agent médical appelle automatiquement les plugins


In [12]:
settings = kernel.get_prompt_execution_settings_from_service_id("openai")
settings.function_choice_behavior = FunctionChoiceBehavior.Auto()
ai_medical_agent.arguments = KernelArguments(settings=settings)

pharmacist_agent = ChatCompletionAgent(
    kernel=kernel,
    name="Pharmacien",
    instructions=PHARMACIST_PROMPT,
)

## Définition d'une stratégie de terminaison

Le parametrage `FunctionChoiceBehavior.Auto()` permet a l'agent `IA_Medicale` d'invoquer automatiquement les plugins enregistres dans le kernel (gravite des symptomes, recommandations). Sans cette configuration, l'agent ne disposerait que de ses instructions textuelles pour formuler ses reponses.

In [13]:
class MedicalTerminationStrategy(TerminationStrategy):
    async def should_terminate(self, agent, history):
        return len(history) >= 6  # On limite à 6 échanges


## Création du chat groupé avec stratégie de terminaison

La classe `MedicalTerminationStrategy` herite de `TerminationStrategy` et implemente `should_terminate`. Ici, la condition est simple : apres 6 messages dans l'historique, le dialogue s'arrete. Cette approche evite les boucles infinies tout en laissant suffisamment d'echanges pour un diagnostic complet.

In [14]:
chat = AgentGroupChat(
    agents=[doctor_agent, ai_medical_agent, pharmacist_agent],
    termination_strategy=MedicalTerminationStrategy()  # Ajout de la stratégie
)

## Fonction pour exécuter le dialogue

La fonction `run_medical_chat` orchestre le dialogue complet : elle recueille les symptomes initiaux via `input()`, transmet le message a l'`AgentGroupChat`, puis itere sur les reponses des agents. Le cycle se poursuit jusqu'a ce que la strategie de terminaison declare la consultation terminee.

En mode interactif (`BATCH_MODE = False`), l'utilisateur peut repondre a chaque agent. En mode batch, l'appel est capture par le bloc `try/except` ci-dessous.

In [15]:
async def run_medical_chat():
    logger.info("🚀 Début de la consultation médicale IA")
    chat_history = ChatHistory()
    
    symptoms = input("Décrivez vos symptômes : ")
    chat_history.add_user_message(symptoms)
    
    while True:
        async for message in chat.invoke():
            logger.info(f"[{message.role}] {message.name}: {message.content}")
            print(f"{message.name}: {message.content}")
            
            if message.name not in ["Pharmacien"]:
                user_response = input("👉 Votre réponse : ")
                chat_history.add_user_message(user_response)
        
        if chat.is_complete:
            break
    
    logger.info("🏥 Consultation terminée.")


In [16]:
try:
    await run_medical_chat()
except (EOFError, KeyboardInterrupt, Exception) as e:
    print(f"[Batch] Consultation interactive ignoree ({type(e).__name__}: {str(e)[:80]})")

2026-05-14 09:21:57,864 [INFO] 🚀 Début de la consultation médicale IA


[Batch] Consultation interactive ignoree (StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.)


## Ce que nous avons construit

Ce cas d'usage illustre plusieurs concepts avances de Semantic Kernel :

| Concept | Implementation |
|---------|----------------|
| **Agents specialises** | Trois roles distincts (Medecin, IA Medicale, Pharmacien) avec instructions systeme dediees |
| **Plugins `@kernel_function`** | Chaque agent dispose d'un plugin avec des fonctions specifiques (questions de suivi, evaluation de gravite, recommandations medicamenteuses) |
| **`AgentGroupChat`** | Orchestration multi-agent avec passage de baton automatique entre les trois participants |
| **Strategie de terminaison** | `MedicalTerminationStrategy` arrete le dialogue apres 6 echanges pour eviter les boucles infinies |
| **`FunctionChoiceBehavior.Auto()`** | L'agent IA Medicale peut appeler automatiquement les plugins du kernel pour enrichir ses reponses |

### Points cles a retenir

1. **Separation des responsabilites** : chaque agent a un role precis, ses propres instructions et ses propres outils. Cette modularite facilite le debug et l'evolution du systeme.

2. **Plugins comme outils** : les `@kernel_function` exposent des capacites que les agents peuvent invoquer via `FunctionChoiceBehavior.Auto()`. Le LLM decide quand et quel plugin appeler en fonction du contexte.

3. **Strategies de controle** : la `TerminationStrategy` est indispensable dans un `AgentGroupChat` pour eviter que les agents ne dialoguent indefiniment. D'autres strategies existent (selection d'agent, filtre de messages).

### Pour aller plus loin

- Ajouter un agent **Patient** simule qui decrit des symptomes de maniere realiste
- Implementer un historique de consultations persistant (stockage dans une base de donnees)
- Utiliser un modele de vecteurs pour rechercher des cas medicaux similaires dans une base de connaissances
- Ajouter des garde-fous ethiques (refus de diagnostic pour certaines pathologies, orientation systematique vers un professionnel de sante)